# EDA: IBM TabFormer credit card transactions

Realistic **synthetic** data from IBM (2,000 people, 6,139 cards, 24.4 million transactions, 1991 to 2020).
Loading only the first 3 million rows (about 252 people, with their complete histories) so it fits in memory.

Why not PaySim: see `01_eda.ipynb`. Almost every sender appears only once, and the balance columns leak the label.

## 1. Load and look

In [1]:
data_path="D:/Coding/fraud-detection-realtime/data/raw/card_transaction.v1.csv"
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/card_transaction.v1.csv", nrows=3_000_000)

**Notes:** what does one row represent? Which column is the target?

## 2. Data quality

In [3]:
df.shape

(3000000, 15)

In [4]:
df.head(10)

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No
5,0,0,2002,9,3,13:53,$86.19,Swipe Transaction,-7146670748125200898,Monterey Park,CA,91755.0,5970,NaN,No
6,0,0,2002,9,4,05:51,$93.84,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
7,0,0,2002,9,4,06:09,$123.50,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
8,0,0,2002,9,5,06:14,$61.72,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
9,0,0,2002,9,5,09:35,$57.10,Swipe Transaction,4055257078481058705,La Verne,CA,91750.0,7538,NaN,No


In [5]:
df.columns=df.columns.str.lower().str.replace(" ","_")

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 15 columns):
 #   Column          Dtype  
---  ------          -----  
 0   user            int64  
 1   card            int64  
 2   year            int64  
 3   month           int64  
 4   day             int64  
 5   time            str    
 6   amount          str    
 7   use_chip        str    
 8   merchant_name   int64  
 9   merchant_city   str    
 10  merchant_state  str    
 11  zip             float64
 12  mcc             int64  
 13  errors?         str    
 14  is_fraud?       str    
dtypes: float64(1), int64(7), str(7)
memory usage: 459.9 MB


In [7]:
df.amount=df["amount"].str.replace("$","").astype(float)

In [8]:
df.columns=df.columns.str.replace("?","")

In [9]:
df[df["amount"]<0].is_fraud.value_counts()

is_fraud
No     143318
Yes       122
Name: count, dtype: int64

In [10]:
df.is_fraud.value_counts(normalize=True)

is_fraud
No     0.99886
Yes    0.00114
Name: proportion, dtype: float64

In [11]:
df["is_fraud"]=(df["is_fraud"]=="Yes").astype(int)

In [12]:
df.head(3)

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud
0,0,0,2002,9,1,06:21,134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,0
1,0,0,2002,9,1,06:42,38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0
2,0,0,2002,9,2,06:22,120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0


In [13]:
df["timestamp"]=df["year"].astype(str)+"-"+df["month"].astype(str)+"-"+df["day"].astype(str)+" "+df["time"]

In [14]:
df["timestamp"]=pd.to_datetime(df["timestamp"],format= "%Y-%m-%d %H:%M")

In [15]:
df["timestamp"].dtype

dtype('<M8[us]')

In [16]:
df["timestamp"].min(), df["timestamp"].max()

(Timestamp('1995-06-18 19:25:00'), Timestamp('2020-02-28 23:49:00'))

In [17]:
df.isna().sum()

user                    0
card                    0
year                    0
month                   0
day                     0
time                    0
amount                  0
use_chip                0
merchant_name           0
merchant_city           0
merchant_state     378407
zip                399736
mcc                     0
errors            2953851
is_fraud                0
timestamp               0
dtype: int64

In [18]:
df[df.duplicated()]

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp
193878,13,0,2015,4,28,06:36,100.0,Chip Transaction,-4282466774399734331,Boyne City,MI,49712.0,4829,"Insufficient Balance,",0,2015-04-28 06:36:00
195221,13,1,2006,7,19,07:34,100.0,Swipe Transaction,-4282466774399734331,Boyne City,MI,49712.0,4829,"Insufficient Balance,",0,2006-07-19 07:34:00
195330,13,1,2006,10,15,07:11,100.0,Swipe Transaction,-4282466774399734331,Boyne City,MI,49712.0,4829,"Insufficient Balance,",0,2006-10-15 07:11:00
244172,17,2,2011,12,12,16:26,120.0,Swipe Transaction,-4282466774399734331,Prescott Valley,AZ,86314.0,4829,"Insufficient Balance,",0,2011-12-12 16:26:00
1360769,109,2,1999,12,9,04:16,120.0,Swipe Transaction,-4282466774399734331,Hammond,IN,46323.0,4829,"Insufficient Balance,",0,1999-12-09 04:16:00
1373587,109,2,2019,3,8,04:22,120.0,Chip Transaction,-4282466774399734331,Hammond,IN,46323.0,4829,"Insufficient Balance,",0,2019-03-08 04:22:00
2469536,205,2,2012,2,8,17:33,82.0,Swipe Transaction,2027553650310142703,Huntington Station,NY,11746.0,5541,NaN,0,2012-02-08 17:33:00
2773484,235,1,2008,5,22,10:23,83.0,Swipe Transaction,1799189980464955940,Garland,TX,75042.0,5499,NaN,0,2008-05-22 10:23:00


In [19]:
df["use_chip"].value_counts()

use_chip
Swipe Transaction     1886840
Chip Transaction       735675
Online Transaction     377485
Name: count, dtype: int64

In [20]:
no_zip=df[df["zip"].isna() & df["merchant_state"].notna()]

In [21]:
no_zip.head()

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp
407,0,0,2003,2,24,13:02,7.52,Swipe Transaction,-727612092139916043,Zurich,Switzerland,NaN,5411,NaN,0,2003-02-24 13:02:00
408,0,0,2003,2,25,06:46,36.97,Swipe Transaction,-4816289482172287511,Zurich,Switzerland,NaN,5411,NaN,0,2003-02-25 06:46:00
906,0,0,2004,7,22,06:32,38.26,Swipe Transaction,3510944833163794547,Tallinn,Estonia,NaN,5912,NaN,0,2004-07-22 06:32:00
907,0,0,2004,7,23,20:05,75.38,Swipe Transaction,8793545055147237096,Tallinn,Estonia,NaN,5812,NaN,0,2004-07-23 20:05:00
908,0,0,2004,7,25,13:08,16.20,Swipe Transaction,2027553650310142703,Tallinn,Estonia,NaN,5541,NaN,0,2004-07-25 13:08:00


In [22]:
no_zip["merchant_state"].value_counts().head(10)

merchant_state
Mexico                6006
Canada                2715
Italy                 1367
United Kingdom        1272
France                 833
China                  765
Germany                753
India                  438
Japan                  421
Dominican Republic     396
Name: count, dtype: int64

## 3. Target

In [23]:
df["is_fraud"].value_counts()

is_fraud
0    2996579
1       3421
Name: count, dtype: int64

In [24]:
df[df["is_fraud"]==1]["user"].nunique()

163

In [25]:
df[df["is_fraud"] == 1].index[0]

np.int64(4099)

In [26]:
df.loc[4099:4099+6]

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp
4099,0,0,2015,11,15,12:55,287.13,Online Transaction,-8194607650924472520,ONLINE,NaN,NaN,3001,NaN,1,2015-11-15 12:55:00
4100,0,0,2015,11,15,13:19,2.41,Online Transaction,-7759074308363763111,ONLINE,NaN,NaN,5651,NaN,1,2015-11-15 13:19:00
4101,0,0,2015,11,16,09:41,50.81,Online Transaction,-551332107213382088,ONLINE,NaN,NaN,4411,NaN,1,2015-11-16 09:41:00
4102,0,0,2015,11,16,09:46,248.36,Online Transaction,4872340518840476610,ONLINE,NaN,NaN,5732,NaN,1,2015-11-16 09:46:00
4103,0,0,2015,11,16,11:20,473.00,Online Transaction,-8566951830324093739,ONLINE,NaN,NaN,3640,NaN,1,2015-11-16 11:20:00
4104,0,0,2015,11,16,11:34,144.49,Online Transaction,-3650057525483423997,ONLINE,NaN,NaN,3007,NaN,1,2015-11-16 11:34:00
4105,0,0,2015,11,16,13:44,44.57,Online Transaction,1913477460590765860,ONLINE,NaN,NaN,5300,NaN,1,2015-11-16 13:44:00


## 4. One column at a time

In [27]:
df["hour"]=df["timestamp"].dt.hour
df["hour"].value_counts().sort_index()

hour
0      23850
1      19329
2      28322
3      21795
4      23953
5      39284
6     175744
7     203535
8     204035
9     195919
10    247426
11    187070
12    197024
13    222028
14    170185
15    168531
16    197414
17    122988
18     92198
19    119044
20     93940
21    113087
22     91300
23     41999
Name: count, dtype: int64

In [28]:
(df.groupby("hour")["is_fraud"].mean())*10000

hour
0      3.354298
1     11.381861
2      8.473978
3     12.388162
4     15.029433
5      7.891253
6      5.291788
7      8.450635
8      9.263117
9     14.342662
10    13.822315
11    18.388838
12    18.627172
13    15.403463
14    18.391750
15    15.012075
16    10.586888
17     9.350506
18     9.870062
19     6.132186
20     3.406430
21     2.387542
22     2.300110
23     2.142908
Name: is_fraud, dtype: float64

In [29]:
(df.groupby("use_chip")["is_fraud"].mean())*10000

use_chip
Chip Transaction       8.114996
Online Transaction    51.657682
Swipe Transaction      4.632083
Name: is_fraud, dtype: float64

In [30]:
df["year"].value_counts().sort_index()

year
1995         1
1996      2068
1997      4838
1998      8154
1999     14526
2000     24388
2001     32404
2002     48102
2003     64797
2004     79823
2005     96725
2006    113868
2007    133518
2008    156374
2009    168643
2010    182862
2011    194377
2012    198262
2013    199765
2014    200781
2015    206503
2016    204288
2017    206275
2018    208983
2019    208618
2020     41057
Name: count, dtype: int64

In [31]:
(df.groupby("year")["is_fraud"].mean() )* 10000

year
1995     0.000000
1996     0.000000
1997     6.200909
1998     0.000000
1999     4.130525
2000    10.660981
2001    12.961363
2002     0.623675
2003     4.475516
2004     3.633038
2005     0.723701
2006    14.490463
2007    17.151246
2008    27.562127
2009     7.352810
2010    23.679059
2011     0.617357
2012     8.070129
2013     9.661352
2014     5.777439
2015    19.612306
2016    19.188596
2017     1.308932
2018    15.503653
2019    12.702643
2020     0.000000
Name: is_fraud, dtype: float64

In [32]:
df.groupby("is_fraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,2996579.0,43.861052,83.730481,-500.0,9.26,30.09,64.08,6820.2
1,3421.0,107.744665,204.257066,-500.0,16.99,69.51,148.58,3750.6


## 5. Feature against target

In [33]:
df["mcc"].head(10)

0    5300
1    5411
2    5411
3    5651
4    5912
5    5970
6    5411
7    5411
8    5411
9    7538
Name: mcc, dtype: int64

In [34]:
df["mcc"].nunique()

109

In [35]:
print(df["mcc"].value_counts().to_string())

mcc
5411    353993
5499    328436
5541    303750
5812    231959
4784    199967
5912    176051
5300    130848
4829    121524
5814    115495
5311    104592
7538    103801
4121     93871
4900     56347
5942     55798
5310     52726
4814     52507
5813     41185
5211     39471
7832     35910
5921     26163
4111     25898
9402     24510
4214     24468
7230     23692
5719     18787
5815     15721
5970     15124
5651     14680
7349     12370
4899     11954
6300     10127
7922      9675
7542      9554
7210      9456
5251      8057
8021      7797
4722      7600
7995      6924
8011      4529
7011      4202
7801      4159
3684      3926
3509      3843
3722      3783
3780      3723
3640      3597
3504      3586
8043      3567
7802      3487
3771      3468
8099      3460
5655      3369
3730      3369
3775      3357
3596      3312
5192      2734
3359      2697
5661      2677
8049      2552
3389      2523
8041      2470
5193      2319
7996      2266
5621      2063
3395      2011
3390      1944
3393  

In [36]:
mcc_tab=df.groupby("mcc")["is_fraud"].agg(["count","mean"])

In [37]:
mcc_tab.head(-1)

,count,mean
mcc,,
1711,674,0.000000
3000,747,0.009371
3001,777,0.011583
3005,93,0.064516
3006,90,0.100000
...,...,...
8049,2552,0.000000
8062,833,0.000000
8099,3460,0.001156


In [38]:
big_sho=mcc_tab[mcc_tab["count"]>=1000]

In [39]:
big_sho.sort_values("mean",ascending=False).head(10)

,count,mean
mcc,,
5094,1019,0.060844
5732,1458,0.056241
5712,1087,0.044158
5932,1059,0.031161
3389,2523,0.011891
7996,2266,0.011474
3509,3843,0.011189
3722,3783,0.010838
5977,1409,0.009936


In [40]:
df["outside_us"]=((df["zip"].isna() )& (df["merchant_state"].notna())).astype(int)

In [41]:
df["outside_us"].value_counts()

outside_us
0    2978671
1      21329
Name: count, dtype: int64

In [42]:
df.groupby("outside_us")["is_fraud"].mean() * 10000

outside_us
0      8.688439
1    390.548080
Name: is_fraud, dtype: float64

In [43]:
df = df.sort_values(["user", "card", "timestamp"])

In [44]:
df["seen_before"]=df.duplicated(subset=["user","card","merchant_name"])

In [45]:
df[["merchant_name","seen_before"]].head(10)

,merchant_name,seen_before
0,3527213246127876953,False
1,-727612092139916043,False
2,-727612092139916043,True
3,3414527459579106770,False
4,5817218446178736267,False
5,-7146670748125200898,False
6,-727612092139916043,True
7,-727612092139916043,True
8,-727612092139916043,True
9,4055257078481058705,False


In [46]:
df["new_shop"]=(~df["seen_before"]).astype(int)
df["new_shop"].value_counts()

new_shop
0    2882245
1     117755
Name: count, dtype: int64

In [47]:
df.groupby("new_shop")["is_fraud"].mean() *10000

new_shop
0      5.547759
1    154.728037
Name: is_fraud, dtype: float64

In [49]:
df["gap"]=df.groupby(["user","card"])["timestamp"].diff()
df[["user","card","timestamp","gap"]].head(10)

,user,card,timestamp,gap
0,0,0,2002-09-01 06:21:00,NaT
1,0,0,2002-09-01 06:42:00,0 days 00:21:00
2,0,0,2002-09-02 06:22:00,0 days 23:40:00
3,0,0,2002-09-02 17:45:00,0 days 11:23:00
4,0,0,2002-09-03 06:23:00,0 days 12:38:00
5,0,0,2002-09-03 13:53:00,0 days 07:30:00
6,0,0,2002-09-04 05:51:00,0 days 15:58:00
7,0,0,2002-09-04 06:09:00,0 days 00:18:00
8,0,0,2002-09-05 06:14:00,1 days 00:05:00
9,0,0,2002-09-05 09:35:00,0 days 03:21:00


In [50]:
df["gap_minutes"]=df["gap"].dt.total_seconds()/60
df[["timestamp","gap","gap_minutes"]].head(10)

,timestamp,gap,gap_minutes
0,2002-09-01 06:21:00,NaT,NaN
1,2002-09-01 06:42:00,0 days 00:21:00,21.0
2,2002-09-02 06:22:00,0 days 23:40:00,1420.0
3,2002-09-02 17:45:00,0 days 11:23:00,683.0
4,2002-09-03 06:23:00,0 days 12:38:00,758.0
5,2002-09-03 13:53:00,0 days 07:30:00,450.0
6,2002-09-04 05:51:00,0 days 15:58:00,958.0
7,2002-09-04 06:09:00,0 days 00:18:00,18.0
8,2002-09-05 06:14:00,1 days 00:05:00,1445.0
9,2002-09-05 09:35:00,0 days 03:21:00,201.0


In [51]:
df.groupby("is_fraud")["gap_minutes"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,2995842.0,1010.793629,2265.410302,0.0,97.0,493.0,1315.0,749489.0
1,3421.0,626.454545,1602.416424,0.0,25.0,117.0,536.0,29690.0


## 6. Weird things and leakage checks

## 7. Findings

### Summary of EDA findings

**Data:** first 3,000,000 rows of IBM TabFormer (about 252 people with complete histories, June 1995 to February 2020). 3,421 frauds, about 11 in 10,000.

**Cleaning done:** lowercase column names; `amount` text to number (removed `$`); `is_fraud` Yes/No to 1/0; `year`, `month`, `day` and `time` joined into `timestamp`; rows sorted by user, card and time.

**Data quality:**
- Missing `errors` means no error. Missing `merchant_state` and `zip` mean an online purchase; missing `zip` with a country in `merchant_state` means a purchase outside the US.
- 8 duplicate rows look like retries after "Insufficient Balance". Kept.
- 143,440 negative amounts are refunds. Not a fraud signal. Kept.

**Clues checked against fraud (frauds per 10,000; overall is about 11):**

| Clue | What was found | Use it? |
|---|---|---|
| `outside_us` | 391 abroad against 9 at home, about 45 times riskier; about a quarter of all fraud | Yes |
| `new_shop` | 155 at a shop the card never used before against 5.5, about 28 times riskier; over half of all fraud | Yes |
| `use_chip` | online 52, chip 8, swipe 5 | Yes |
| `mcc` | jewellery 608, electronics 562, furniture 442, antiques 312; groceries and fuel very low | Yes, as a category |
| `gap_minutes` | middle gap since the card's last purchase is about 2 hours for fraud against 8 hours for normal | Yes |
| `amount` | middle fraud purchase $70 against $30 for normal | Yes |
| `hour` | highest from 11 am to 2 pm (about 19), lowest from 8 pm to midnight (about 2) | Yes |
| `year` | jumps between 0 and 28 because fraud comes in bursts and there are few people | No |

**Not used as features:** `year`, `user`, `card`, raw `timestamp`, `merchant_name` (only used to build `new_shop`).

**Things to handle before training:**
- Split by time, not randomly, because fraud comes in bursts on the same card.
- `mcc` and `use_chip` are categories and need encoding.
- Each card's first purchase has no `gap_minutes`.
- Use the same time zone for training and live scoring.
- Judge the model with precision, recall and PR-AUC, not accuracy.
